In [1]:
# Cohort Creation TBI Having EPI
#Step1 
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb

util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [ ]:
TBI_with_EPI_Total = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_with_EPI_cohort1')

In [ ]:
TBI_with_EPI_Total.createOrReplaceTempView('TBI_EPI_Cohort1')

In [ ]:
TBI_EPI_1_result = spark.sql("SELECT COUNT(DISTINCT (personid)) FROM Cohort1_Table")
TBI_EPI_1_result.show()

In [ ]:
TBI_with_EPI_Total.printSchema()

In [2]:
#Step2
TBI_with_EPI_Total_Final = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_cohort')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Step3
TBI_with_EPI_Total_Final.printSchema()

In [3]:
#Step4
TBI_with_EPI_Total_Final.createOrReplaceTempView('TBI_EPI')

▸,:,


In [ ]:
TBI_EPI_ResultView = spark.sql("""select * from TBI_EPI where conditioncode = 'Z87.820'""")
TBI_EPI_ResultView.show(truncate = False)

In [ ]:
TBI_with_EPI_Total_Final.show(5)

In [ ]:
tables = spark.catalog.listTables('real_world_data_jun_2022')
table_names = [table.name for table in tables]
print(table_names)

In [ ]:
table_schema = spark.sql(f"DESCRIBE {'real_world_data_jun_2022'}.{'medication'}")

# Show the schema
table_schema.show(truncate=False)

In [ ]:
#Extract distinct personids from TBI_EPI_Cohort1
distinctPersonIDsCohort1 = TBI_with_EPI_Total.select("personid").distinct()

#Extract distinct personids from TBI_EPI_Cohort
distinctPersonIDsCohort = TBI_with_EPI_Total_Final.select("personid").distinct()

#Find the personids that are in TBI_EPI_Cohort but not in TBI_EPI_Cohort1
personIDsNotInCohort = distinctPersonIDsCohort.subtract(distinctPersonIDsCohort1)

#Count the distinct personids not in TBI_EPI_Cohort1
countNotInCohort = personIDsNotInCohort.count()

#Print the count
print("Count of distinct personids not in TBI_EPI_Cohort1:", countNotInCohort)

In [ ]:
TBI_with_EPI_Total_Final.printSchema()

In [ ]:
#Step5
EPI = spark.sql("""
select * from TBI_EPI
where conditioncode like 'G40%' or
conditioncode like '345%'
""")

In [ ]:
EPI.createOrReplaceTempView('EPI_Table')

In [ ]:
EPI.printSchema()

In [ ]:
EPI_Result = spark.sql("""
select DISTINCT personid, COUNT(*) As count from EPI_Table GROUP BY personid HAVING count > 2
""")
EPI_Result.show(5, truncate = False)

In [ ]:
#Step6
TBI = spark.sql("""
select * from TBI_EPI
where conditioncode in ('Z87.820','S02.1','R56.1') or
conditioncode like 'S06%' or
conditioncode like 'S07%' or
conditioncode like 'S08%' or
conditioncode like 'S09%' or
conditioncode like 'G44.3%' or
conditioncode like '854.%' or
conditioncode like '851.%' or
conditioncode like '852.%' or
conditioncode like '853.%'
""")

In [ ]:
EPI_med = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/epi_med')

In [ ]:
EPI_med.printSchema()

In [ ]:
EPI_med.createOrReplaceTempView('EPI_Med_Table')

In [ ]:
#Extract distinct personids from EPI_med
distinctPersonIDsCohort1 = EPI_med.select("personid").distinct()

#Extract distinct personids from TBI_EPI_Cohort
distinctPersonIDsCohort = TBI_with_EPI_Total_Final.select("personid").distinct()

#Find the personids that are in EPI_med but not in TBI_EPI_Cohort
personIDsNotInCohort = distinctPersonIDsCohort1.subtract(distinctPersonIDsCohort)

#Count the distinct personids not in TBI_EPI_Cohort
countNotInCohort = personIDsNotInCohort.count()

#Print the count
print("Count of distinct personids not in TBI_EPI_Cohort:", countNotInCohort)

In [ ]:
TBI_EPI_Med_result = spark.sql("""
    SELECT e.personid, e.startdate AS date
    FROM EPI_Med_Table e
    LEFT JOIN TBI_EPI t ON e.personid = t.personid
    WHERE t.personid IS NULL
""")
TBI_EPI_Med_result.show(5, truncate = False)

In [ ]:
TBI_EPI_Med_result.createOrReplaceTempView('TBI_EPI_Med')

In [ ]:
TBI_EPI_Med_Extract = spark.sql("""
    SELECT COUNT(DISTINCT(personid)) from TBI_EPI_Med
""")
TBI_EPI_Med_Extract.show()

In [ ]:
TBI_EPI_Med_result.write.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_Med_Final.parquet")

In [ ]:
EPI_total_cohort = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/EPI_total_cohort')

In [ ]:
EPI_total_cohort.createOrReplaceTempView('EPI_total_cohort_table')

In [ ]:
#Extract distinct personids from EPI_total_cohort
distinctPersonIDsCohort1 = EPI_total_cohort.select("personid").distinct()

#Extract distinct personids from TBI_EPI_Cohort
distinctPersonIDsCohort = TBI_with_EPI_Total_Final.select("personid").distinct()

#Find the personids that are in EPI_med but not in TBI_EPI_Cohort
personIDsNotInCohort = distinctPersonIDsCohort1.subtract(distinctPersonIDsCohort)

#Count the distinct personids not in TBI_EPI_Cohort
countNotInCohort = personIDsNotInCohort.count()

#Print the count
print("Count of distinct personids not in TBI_EPI_Cohort:", countNotInCohort)

In [ ]:
TBI_EPI_result = spark.sql("SELECT COUNT(DISTINCT (personid)) FROM TBI_EPI")
TBI_EPI_result.show()

In [ ]:
med_result = spark.sql("SELECT COUNT(DISTINCT (personid)) FROM EPI_Med_Table")
med_result.show()

In [ ]:
TBI_EPI_Med_result = spark.sql("SELECT e.personid, COUNT(*) as count FROM TBI_EPI t right join EPI_Med_Table e on t.personid = e.personid where e.personid is NULL GROUP BY e.personid")
TBI_EPI_Med_result.show(5, truncate = False)

In [ ]:
TBI_EPI_Cohort_Demo = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_demo')

In [ ]:
TBI_EPI_Cohort_Demo.printSchema()

In [ ]:
TBI_EPI_Cohort_Demo.show(5)

In [ ]:
#2
TBI_EPI_Cohort_Demo.createOrReplaceTempView('TBI_EPI_Cohort_Demo')

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

In [ ]:
result = spark.sql("SELECT DISTINCT personid, COUNT(*) as count FROM TBI_EPI_Cohort_Demo GROUP BY personid HAVING count > 4")
result.show(5, truncate = False)

In [ ]:
result1 = spark.sql("SELECT * FROM TBI_EPI_Cohort_Demo where personid = '8f6ea322-a652-49e2-8007-28310a5fe0ec'")
result1.show(5, truncate = False)

In [ ]:
DemoTab = spark.sql("SELECT * FROM demographics where personid = '8f6ea322-a652-49e2-8007-28310a5fe0ec'")
DemoTab.show(5, truncate = False)

In [ ]:
EPI_total = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/EPI_total_cohort')

In [ ]:
EPI_total.printSchema()

In [ ]:
TBI_total = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_total_cohort')

In [ ]:
TBI_total.printSchema()

In [ ]:
#Step7
# Extract the person IDs from both DataFrames
# epi_personids = EPI.select("personid")
tbi_personids = TBI.select("personid").distinct()
count = tbi_personids.count()
print("Number of distinct personids: ", count)

# Find the intersecting person IDs
# intersecting_personids = epi_personids.intersect(tbi_personids).distinct()

# # Show the intersecting person IDs
# intersecting_personids.show()

In [ ]:
#Step7
# Extract the person IDs from both DataFrames
epi_personids = EPI.select("personid")
tbi_personids = TBI.select("personid")

# Find the intersecting person IDs
intersecting_personids = epi_personids.intersect(tbi_personids).distinct()

# # Show the intersecting person IDs
# intersecting_personids.show()

In [ ]:
#Step8
# Extract the person IDs from both DataFrames
epi_personids = EPI.select("personid").distinct()
tbi_personids = TBI.select("personid").distinct()

# Find the TBI person IDs but not EPI by subtracting EPI
subtracting_personids = tbi_personids.subtract(epi_personids).distinct()
count = subtracting_personids.count()
print("Number of distinct personids: ", count)
# # Show the intersecting person IDs
# intersecting_personids.show()

In [ ]:
subtracting_personids.createOrReplaceTempView('TBI_NOT_EPI_Cohort')

In [ ]:
TBI_NOT_EPI_Extract = spark.sql("""
    SELECT COUNT(DISTINCT(personid)) from TBI_NOT_EPI_Cohort
""")
TBI_NOT_EPI_Extract.show()

In [ ]:
#Step8 -> TBI Not Having(subtract)EPI Control with diagnosis date as date
# Write the distinct subtracting person IDs to a Parquet file
subtracting_personids.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_NOT_EPI_Cohort_PersonId_Final")

In [ ]:
intersecting_personids.createOrReplaceTempView('TBI_EPI_Cohort')

In [ ]:
TBI_EPI_Extract = spark.sql("""
    SELECT COUNT(DISTINCT(personid)) from TBI_EPI_Cohort
""")
TBI_EPI_Extract.show()

In [ ]:
#Step8 -> TBI Having(Intersect)EPI Cohort with diagnosis date as date
# Write the distinct intersecting person IDs to a Parquet file
intersecting_personids.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_Cohort_PersonId_Final")

In [4]:
#Step9 - Read TBI_EPI_Cohort
TBI_EPI_Cohort_PersonId_Final = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_Cohort_PersonId_Final')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
TBI_EPI_Cohort_PersonId_Final.printSchema()

In [5]:
TBI_EPI_Cohort_PersonId_Final.createOrReplaceTempView('TBI_with_EPI_PersonId')

▸,:,


In [ ]:
TBI_with_EPI_Total_Final.printSchema()

In [6]:
TBI_with_EPI_Total_Final.createOrReplaceTempView('TBI_EPI')

▸,:,


In [ ]:
TBI_with_EPI_all = spark.sql("SELECT r.personid, t.date as date FROM TBI_EPI t inner join TBI_with_EPI_PersonId r where t.personid = r.personid")

In [ ]:
TBI_with_EPI_all.printSchema()

In [ ]:
TBI_with_EPI_all.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_Cohort_Final_V1")

In [ ]:
cohort_demo = etl.extractDemo(spark,intersecting_personids,outputfilename="cohort_demo", outputfolder = "Priya/epilepsy")

In [ ]:
from pyspark.sql.functions import *
demo_processed = etl.process_demo(spark, intersecting_personids, cohort_demo)

In [7]:
#TBI Having EPI - Read the Data
#Step2 - Read TBI_EPI_Cohort
TBI_EPI_Cohort_PersonId_Final = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_Cohort_PersonId_Final')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
TBI_EPI_Cohort_PersonId_Final.createOrReplaceTempView('TBI_EPI_Cohort')

▸,:,


In [9]:
TBI_EPI_Cohort_Result = spark.sql("""
  SELECT t.personid, t.date
  FROM TBI_EPI t
  LEFT JOIN TBI_EPI_Cohort e ON t.personid = e.personid
  WHERE e.personid IS NOT NULL
""")

TBI_EPI_Cohort_Result.show(5, truncate = False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------------------+
|personid                            |date                     |
+------------------------------------+-------------------------+
|b18b89e1-a4c7-4167-b4ab-68b93e39c2c8|2019-01-11T08:00:00+00:00|
|1928edb4-84e1-4682-90a3-35a277e3fe3d|2017-08-25T07:00:00+00:00|
|1928edb4-84e1-4682-90a3-35a277e3fe3d|2017-11-01T08:00:00+00:00|
|1928edb4-84e1-4682-90a3-35a277e3fe3d|2017-11-01T08:00:00+00:00|
|1928edb4-84e1-4682-90a3-35a277e3fe3d|2017-11-01T08:00:00+00:00|
+------------------------------------+-------------------------+
only showing top 5 rows



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
TBI_EPI_Cohort_Result.createOrReplaceTempView('TBI_EPI_Cohort_Final')
# TBI_EPI_Cohort_ResultCount = spark.sql("""
#   SELECT COUNT(personid) as count_of_records FROM TBI_EPI_Cohort_Final
# """)

# TBI_EPI_Cohort_ResultCount.show()

▸,:,


In [ ]:
TBI_EPI_Cohort_Update = spark.sql("""
  SELECT personid, min(date) as date FROM TBI_EPI_Cohort_Final group by personid
""")

TBI_EPI_Cohort_Update.show()

In [11]:
TBI_EPI_Cohort_Update1 = spark.sql("""
  SELECT personid, COALESCE(min(date), NULL) as date
  FROM TBI_EPI_Cohort_Final
  GROUP BY personid
""")

TBI_EPI_Cohort_Update1.show()

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------------------+--------------------+
|            personid|                date|
+--------------------+--------------------+
|0137680e-a2a0-453...|2020-09-30T19:16:...|
|0185fdd7-4c5b-451...|                    |
|025ff86d-bea8-452...|                    |
|030160ee-4cda-49f...|2016-12-07T05:37:...|
|032145cc-8b8d-41b...|                    |
|03be950e-ad98-433...|2020-05-26T12:41:...|
|044efcb9-9e28-4f1...|2017-07-15T04:00:...|
|04860a66-a13b-45b...|                    |
|04a07847-d106-4cd...|2015-10-28T01:57:...|
|06b4a13d-ca4a-411...|2016-06-24T07:00:...|
|06d1c431-8624-45a...|2018-04-20T07:00:...|
|072d8a1f-f03f-42d...|2015-08-14T19:30:...|
|07af9797-9dc3-437...| 2016-11-12T00:00:00|
|07c2a844-67a9-49f...|2011-12-05T13:02:...|
|08ce5b8d-6828-4fb...|2012-04-03T01:50:...|
|08dbf3fb-60c5-4f2...|                    |
|09239493-682c-452...|                    |
|09c0c316-029d-41f...|                    |
|09df884e-f178-4c0...|                    |
|0a47976e-19a6-45b...|          

<IPython.core.display.Javascript object>

In [14]:
TBI_EPI_Cohort_Update1.createOrReplaceTempView('TBI_EPI_Cohort_Updated')

▸,:,


In [15]:
TBI_EPI_Cohort_UpdatedCount = spark.sql("""
  SELECT COUNT(personid) as count_of_records FROM TBI_EPI_Cohort_Updated
""")

TBI_EPI_Cohort_UpdatedCount.show()

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+----------------+
|count_of_records|
+----------------+
|          102687|
+----------------+



<IPython.core.display.Javascript object>

In [13]:
TBI_EPI_Cohort_Update1.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_Cohort_Final_Updated_1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>